# 第8回 演習：分類（解答例・教員用）

## 今日の分析目標

**心臓病かどうかを、見逃しを抑えて判定したい。**

この演習では、LogisticRegression で心臓病である確率を出し、混同行列・適合率・再現率で評価し、しきい値を下げて見逃しを減らす——この一連の流れを、心臓病データで自分の手でたどります。しきい値を自分の手で動かして「見逃しを減らす」感覚をつかみましょう。TODOに取り組みながら、最後の「目標に答えられたか」で振り返りましょう。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_curve, auc, recall_score, precision_score)

!pip install -q japanize-matplotlib   # 図中の日本語が □ になるのを防ぐ
import japanize_matplotlib

plt.rcParams['figure.figsize'] = (7, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の化けを防ぐ
DATA_DIR = 'https://raw.githubusercontent.com/k0heiun0/applied_exercise/main/shared/data'   # データはこのリポジトリから読み込む
df = pd.read_csv(f'{DATA_DIR}/heart.csv')
feat = [c for c in df.columns if c != 'target']
X, y = df[feat].values, df['target'].values
def pipe(model):
    return Pipeline([('imp', SimpleImputer(strategy='median')),
                     ('sc', StandardScaler()), ('m', model)])
print(f'{X.shape[0]}件, 心臓病あり{y.sum()}人 / なし{(y==0).sum()}人')

## 1. モデルを比べる

LogisticRegression と LDA を、交差検証の正解率で比べます。

### 深掘り：なぜ「回帰の直線」ではダメか——線形スコアをシグモイドで確率に写す

当てたいのは `target`＝「心臓病あり(1)／なし(0)」の二択です。第2回からの回帰をそのまま使い、$\hat y = \beta_0 + \sum_j \beta_j x_j$ という**直線**で 0/1 を予測してはいけないのでしょうか。試すと、うまくいかない理由がはっきり出ます。この心臓病データで普通の線形回帰を当てはめ、テストの61人を予測させると、出てくる値は最小 **−0.18**・最大 **1.34** まで散らばり、**61人中9人が 0〜1 の外**にはみ出します。確率が「マイナス」や「1超え」になるのは意味をなしません。直線は上にも下にも限りなく伸びるので、そもそも**確率の器（0〜1）に収まらない**のです。しかも二乗誤差で当てはめる直線は、遠くの点に引っぱられて境目がずれやすい、という弱点もあります。

**ロジスティック回帰の骨子**　そこで一工夫します。第2回の重みづけの足し算——線形スコア $z = \beta_0 + \sum_j \beta_j x_j$——はそのまま作り、最後にそれを**シグモイド関数**という S 字カーブに通します。

$$
p = \sigma(z) = \frac{1}{1 + e^{-z}}, \qquad z = \beta_0 + \sum_{j=1}^{p} \beta_j x_j
$$

$\sigma(z)$ は、どんな $z$ を入れても必ず **0 と 1 のあいだ**の値を返します。記号をほどくと——$z$ が大きくプラスなら $e^{-z}\to 0$ で $\sigma \to 1$（ほぼ確実に病気）、大きくマイナスなら $e^{-z}\to\infty$ で $\sigma \to 0$（ほぼ健康）、ちょうど $z=0$ のとき $\sigma = 1/2 = 0.5$。こうして**青天井の線形スコアを、確率という器へ写す**のがシグモイドの役目です。$p = \sigma(z)$ は「この患者が心臓病である確率」と読めます。各検査値 $x_j$ にかかる係数 $\beta_j$ の意味も回帰と地続きで、$\beta_j$ が正ならその検査値が上がるほどスコア $z$ が上がって病気側へ、負なら健康側へ、確率を押します。

**判定と決定境界**　確率が出たら、$p > 0.5$ なら「心臓病あり」と判定します。$\sigma(z) > 0.5$ は $z > 0$ と同じことなので、判定が切り替わる境目 $p = 0.5$ は $z = 0$、すなわち $\beta_0 + \sum_j \beta_j x_j = 0$ という**まっすぐな線（面）**になります。これが**決定境界**で、LogisticRegression が「線形」の分類器と呼ばれる理由です。名前に「回帰」とありますが、中身は**分類**の手法だ、という点に注意してください。

**この節の実データ**　1節では LogisticRegression と LDA を交差検証の正解率で比べます。結果はそれぞれ **0.828** と **0.832**——どちらも約 0.83 で、ほぼ互角です。LDA（線形判別分析）も理屈は違えど、まっすぐな境界で分ける線形分類器で、「2つのクラスがいちばんよく分かれて見える軸」を探すという別の考え方から出発しますが、結果は似ることが多いのが特徴です。なお 8割強という**程よい難しさ**は大事なポイントで、これから見るしきい値や見逃しの話は、正解率がほぼ 1.0 になる易しすぎるデータでは成り立ちません。この心臓病データが「そこそこ間違える」からこそ、見逃しを数え、しきい値で調整する意味が出てきます。

**つまずき：なぜ先に標準化するのか**　この演習の `pipe()` は、補完のあと `StandardScaler`（平均0・ばらつき1に揃える）をはさんでからモデルに渡しています。省くとどうなるか。線形スコア $z = \sum_j \beta_j x_j$ は各検査値の**単位のスケール**に左右されます。年齢（数十のオーダー）とコレステロール（数百のオーダー）を生のまま混ぜると、値の大きい変数だけがスコアを支配しがちで、係数の読みも学習の収束も不安定になります。スケールを揃えておけば各検査値が対等な土俵に乗り、係数どうしを見比べられます。標準化を忘れる——これは分類でありがちな落とし穴の一つです。

**もっと踏み込みたい人へ**　直線回帰が予測の外れの二乗和（最小二乗）で係数を決めたのに対し、ロジスティック回帰の係数は「観測された 0/1 がいちばん起こりやすくなるように」——**最尤法**で決めます。この最尤推定や、$\log\frac{p}{1-p} = z$（ロジット＝対数オッズ）が線形になるという式変形など厳密な数理は、**詳しくは MVA『ロジスティック回帰』回**へ譲ります。ここでは「線形スコアをシグモイドで確率に写す」骨組みをつかめば十分です。

In [ ]:
for name, model in [('LogisticRegression', LogisticRegression(max_iter=5000)),
                    ('LDA', LinearDiscriminantAnalysis())]:
    s = cross_val_score(pipe(model), X, y, cv=5, scoring='accuracy')
    print(f'{name:20s}: 正解率 {s.mean():.3f}')

## 2. 混同行列で「見逃し」を数える

### TODO①：テストデータで混同行列を作る

`Xtr, Xte, ytr, yte`（下で用意済み）を使い、LogisticRegression のパイプラインを学習して、テストデータの混同行列を作ってください。見逃し（本当は病気なのに健康と判定）が何件かを確認しましょう。

### 深掘り：混同行列の四つのマスと、医療で重い「見逃し」

正解率は「全体で何割当たったか」の一つの数字ですが、それだけでは**何をどう間違えたか**が見えません。病気を見逃したのか、健康な人を騒がせたのか——中身が潰れてしまうのです。そこで `confusion_matrix(真値, 予測)` を使います。この関数は**行が真値・列が予測**の表を返し、ここでは陽性（positive）を **心臓病（target=1）** に取ります。すると予測と真実の組み合わせが四つのマスに分かれます。

- **真陽性 TP**（右下）：本当に病気で、病気と当てた人（正しく陽性）
- **真陰性 TN**（左上）：本当に健康で、健康と当てた人（正しく陰性）
- **偽陽性 FP＝誤検出**（右上）：健康なのに病気と判定した人
- **偽陰性 FN＝見逃し**（左下）：**本当は病気なのに健康と判定した人**

対角線（TP と TN）が正解、非対角のマス（FP と FN）が二種類の間違いです。TODO① をテストデータ（61人、内訳は心臓病 **28人**・健康 **33人**）で解くと、しきい値 0.5 での混同行列は次の数になります——**TN=27, FP=6, FN=2, TP=26**。つまり心臓病を正しく拾えた人が **26人**、誤検出（健康を病気と誤り）が **6人**、そして**見逃し（左下）が 2人**です。TODO①のヒートマップで、左下のマスに 2 が立つのを確かめてください。行が真値・列が予測、という向きを取り違えると見逃しと誤検出が入れ替わってしまうので、軸ラベルを必ず確認する癖をつけましょう。

**なぜ医療では左下が重いのか**　同じ「1件の間違い」でも、二種類の重みはまるで違います。**見逃し（FN）**は、本当に病気の患者を「健康」と帰してしまうこと——治療の機会を失い、命に関わりかねません。いっぽう**誤検出（FP）**は、健康な人を「病気かも」と拾ってしまうこと——多くは再検査で済みます。**間違いのコストが非対称**なのです。だから医療では、四つのマスのうち**左下の見逃しにいちばん注目**し、これをどう減らすかを考えます。ここが「正解率という一つの数字」では絶対に見えない部分で、混同行列を起点にする理由そのものです。どちらを陽性に置くかで指標の意味が丸ごと変わるので、以降つねに「陽性＝心臓病」を軸に読み進めてください。

**別の見方：この四マスはどんな分類器でも同じ**　混同行列は LogisticRegression 専用の道具ではありません。1節で比べた LDA でも、あるいは今回は使わない決定木・ランダムフォレスト・サポートベクターマシンといった**曲がった境界を引く手法**でも、出てくるのは結局「病気を病気と当てたか／健康を病気と誤ったか」の四マスです。モデルの中身がどれだけ複雑でも、それを測る**ものさし（混同行列・適合率・再現率・後で出る ROC）は共通**なのです。だからこの節で身につける読み方は、この先どんな分類器に持ち替えてもそのまま効きます。

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
clf = pipe(LogisticRegression(max_iter=5000)).fit(Xtr, ytr)

# TODO: テストデータ（Xte, yte）で混同行列を作って表示してください

# 解答例①：混同行列
cm = confusion_matrix(yte, clf.predict(Xte))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['健康', '心臓病'], yticklabels=['健康', '心臓病'])
plt.xlabel('予測'); plt.ylabel('真値'); plt.tight_layout(); plt.show()
print('左下が見逃し =', cm[1, 0], '件')


## 3. 適合率・再現率・F1

答え合わせも兼ねて、指標の一覧を出します。心臓病の再現率（見逃しの少なさ）に注目してください。

### 深掘り：適合率・再現率・F1 の骨子と、「正解率だけ」の罠

四つのマスが数えられれば、目的別の指標が計算できます。陽性は引き続き**心臓病**です。骨子は次の二本です。

$$
\text{適合率（precision）} = \frac{TP}{TP + FP}, \qquad
\text{再現率（recall）} = \frac{TP}{TP + FN}
$$

- **適合率**は「病気と**判定した**うち、本当に病気だった割合」——分母が予測陽性（$TP+FP$）なので、**誤検出（FP）の少なさ**を測ります。適合率が低い＝空振りの陽性判定が多い、ということです。
- **再現率**は「本当に病気の人のうち、拾えた割合」——分母が真の陽性（$TP+FN$）なので、**見逃し（FN）の少なさ**を測ります。再現率が低い＝取りこぼしが多い、ということです。

同じ TP を使っても**何で割るか**が違うので、見ている中身が別物だという点が肝心です。前節の実数（TP=26, FP=6, FN=2）を入れると、適合率 $= 26/(26+6) = 26/32 = $ **0.81**、再現率 $= 26/(26+2) = 26/28 = $ **0.93**。3節の `classification_report` の心臓病の行がこの値（適合率 0.81・再現率 0.93）と一致します。再現率 0.93 は「真の病気の 93% を拾えた」＝残り 7%（＝見逃し2人）を取りこぼした、と読めます。医療の目標は見逃しを避けることなので、ここでは**再現率を重く見ます**。逆に「誤って病気と騒ぎたくない」場面（たとえば高額な追加検査が伴うとき）なら適合率を重く見る、と目的しだいで軸が変わります。

適合率と再現率は両立しにくく（片方を上げると片方が下がりがち）、その総合点が **F1** です。二つの**調和平均**で、$F_1 = \dfrac{2 \cdot \text{適合率} \cdot \text{再現率}}{\text{適合率} + \text{再現率}}$。ふつうの平均でなく調和平均を使うのは、**どちらか一方だけ高くてもスコアが伸びない**ようにするためです（片方が 0 に近いと F1 も 0 に近づく）。両方そろって高いときだけ F1 も高くなる。今回は約 **0.87** で、適合率 0.81 と再現率 0.93 のあいだのバランス点に落ち着いています。

**`classification_report` を全部読む**　3節の表は心臓病の行だけでなく、**健康の行**も出しています。健康クラスの適合率は **0.93**、再現率は **0.82**——心臓病クラス（適合率 0.81・再現率 0.93）と、ちょうど**鏡写し**になっているのが分かります。これは偶然ではありません。心臓病の見逃し（FN=2）は健康側から見れば「健康と当てた中の取りこぼし」ではなく、逆に心臓病の誤検出（FP=6）は健康側から見れば「健康なのに拾えなかった」分にあたるため、一方の適合率が他方の再現率と絡み合うのです。表の下にある `macro avg`（二クラスの単純平均）や `accuracy`（0.87）も一緒に見ると、モデルが片方のクラスだけ得意になっていないかを確かめられます。どの一行を見るかは、けっきょく「どちらの間違いを重く見るか」という目標しだいです。

**つまずき：正解率だけを見る危険**　いちばん素朴な指標は正解率 $=(TP+TN)/\text{全体}$ で、今回は $(26+27)/61 = $ **0.87**。悪くない数字ですが、正解率は**クラスが偏ると簡単にだまされます**。たとえば病気が全体の 1% しかいないデータで、「**全員健康**」と答えるだけの、何も考えていないモデルを考えると、正解率は **0.99** になります。ところがこのモデルの再現率は **0.00**——病気を**1件も拾えていません**。正解率だけ見ていれば「99%の名医」に見えるのに、肝心の病気は全滅、というわけです。だからこそ混同行列を起点に、再現率・適合率・F1 で多角的に見ます。幸い今回の心臓病データは病気 139・健康 164 とほぼ半々（陽性が約46%）なので正解率もそれなりに使えますが、不正検知・故障予測・レアな病気の診断など**珍しい事象を当てる不均衡データ**では、この罠がつねに口を開けています。「数字一つにだまされない」——第5回・第6回と同じ教訓が、分類でもそのまま効いています。

In [ ]:
print(classification_report(yte, clf.predict(Xte), target_names=['健康', '心臓病']))

## 4. しきい値を下げて、見逃しを減らす

### TODO②：しきい値0.3での再現率

`predict_proba` で心臓病である確率を出し、**しきい値0.3**で判定したときの再現率と適合率を計算してください。しきい値0.5のときと比べて、見逃しはどうなりますか？

### 深掘り：しきい値というつまみ——見逃しと誤検出のトレードオフ

ここまでは「確率 0.5 を超えたら病気」と判定してきました。でもこの **0.5 は絶対の値ではなく、動かせるつまみ**です。`predict_proba` は一人ひとりの心臓病確率 $p = \sigma(z)$ を返します（テスト先頭5人なら 0.22, 0.71, 0.06, 0.04, 0.54 といった値）。確率 0.99 の陽性と 0.51 の陽性では、同じ「病気」でも**自信のほどがまるで違う**——この濃淡こそ確率を出せる手法の強みです。この $p$ をどの値で区切るか、それがしきい値です。

見逃しを減らしたいなら、しきい値を**下げて**「疑わしきは病気に」と広めに拾います。TODO② で、しきい値を 0.5 → 0.3 と下げると、指標は次のように動きます。

| しきい値 | 再現率 | 適合率 | 見逃し FN | 誤検出 FP |
|---:|---:|---:|---:|---:|
| 0.5 | 0.93 | 0.81 | 2 | 6 |
| 0.4 | 0.93 | 0.81 | 2 | 6 |
| 0.3 | **1.00** | 0.70 | **0** | 12 |

しきい値 0.3 まで下げると**再現率 1.00＝見逃しゼロ**。そのかわり誤検出は 6→12 と倍増し、適合率は 0.81→0.70 に落ちます。**見逃しを消せば誤検出が増える**——これが避けられないトレードオフです。医療では見逃しのほうが痛いので、多少の誤検出増を受け入れてでもしきい値を下げる、という判断があり得ます。逆に区切りを上げれば、病気判定は減って適合率は上がるかわり、見逃しが増える。どこで切るのが正解かはデータだけでは決まらず、**目標が、どの指標を優先しどこで区切るかを決める**わけです。しきい値を 0.5 に固定するのは単なる初期設定にすぎない、と覚えておいてください。

一つ細かいけれど大事な観察を。0.5 と 0.4 で数字が**まったく同じ**なのに気づきましたか。しきい値は連続に効くのではなく、**実在する誰かの確率をまたいで初めて**判定が動きます。テスト患者の確率が (0.4, 0.5] の区間に一人もいなければ、そのあいだしきい値を動かしても混同行列は 1 ミリも変わりません。0.3 まで下げて初めて何人かの確率をまたぎ、見逃しが 2→0、誤検出が 6→12 と一気に動いたのです。だから確率の分布しだいで、しきい値を下げても「効く区間」と「効かない区間」がある、という感覚を持っておくと役立ちます。なおこの確率の濃淡は、「確率の高い人はすぐ治療へ、グレーの人だけ精密検査へ」といった運用の使い分けにもそのまま活きます。

In [ ]:
prob = clf.predict_proba(Xte)[:, 1]   # 心臓病である確率

# TODO: しきい値0.3で判定した予測を作り、再現率と適合率を計算して表示してください

# 解答例②：しきい値0.3の再現率・適合率
for t in [0.5, 0.3]:
    p = (prob > t).astype(int)
    print(f'しきい値{t}: 再現率 {recall_score(yte, p):.2f} / 適合率 {precision_score(yte, p):.2f}')
print('→ しきい値を下げると再現率が上がり（見逃しが減り）、適合率は下がる')


## 5. ROC曲線とAUC

しきい値を全部試したときの性能を、ROC曲線とAUCで見ます。

### 深掘り：ROC 曲線と AUC——しきい値を全部試す

前節では 0.5・0.4・0.3 と、しきい値を手で三つ試しました。ならば **0 から 1 まで全部試したら**どうなるか——それを一枚に描いたのが **ROC 曲線**です。しきい値を高いほうから下げながら、各しきい値での二つの率を点として打ち、その軌跡をつなぎます。

- 横軸 **FPR（誤検出率）** $= \dfrac{FP}{FP+TN}$：健康な人を誤って病気と言った割合
- 縦軸 **TPR（検出率）** $= \dfrac{TP}{TP+FN}$：これは**再現率そのもの**——病気を拾えた割合

しきい値を下げるほど点は**右上へ**進みます（病気を拾う TPR が上がると同時に、誤検出 FPR も上がる）。前節のしきい値 0.5・0.3 での (FPR, TPR) が、この曲線上の二つの点にあたる、と考えると図と表がつながります。理想は**左上の角**（検出率 1・誤検出率 0＝見逃しも誤検出もゼロ）で、曲線が左上に張り付くほど優秀です。対角線 $y=x$ は「でたらめ（コイン投げ）」の目安で、そこより上にあれば当てずっぽうよりマシ、ということになります。

**AUC**（Area Under the Curve）は、この曲線の**下の面積**です。1.0 で完璧、0.5 で対角線＝でたらめ。直感的には「**ランダムに選んだ病気の人が、ランダムに選んだ健康な人より高い確率を付けられる確率**」で、モデルが病気を健康より上位に並べる**順位づけの良さ**を表します。5節で描くとおり、この心臓病分類の AUC は **0.95**。実データとしては十分に高く、モデルが病気と健康をよく並べ分けられていることを意味します（乳がんのようにほぼ 1.0 とはいかないのは、このデータが程よく難しいからです）。

AUC の利点は、**しきい値を一つに決めなくても測れる**こと。前節で見たようにしきい値ごとに再現率も適合率も動くので、一つのしきい値での成績だけでモデルの優劣を語ると、区切り位置に振り回されます。AUC はしきい値を全部試した全体像を一つの数字に要約するので、**モデルどうしの実力比較**に向いた物差しになります。ただし面積という一つの数字に潰す以上、「見逃しをとくに重く見たい」といった目的の非対称性は AUC には映りません。最終的にどのしきい値で運用するかは、やはり混同行列に戻って、見逃しと誤検出の重みを見比べて決めることになります。

In [ ]:
fpr, tpr, _ = roc_curve(yte, prob)
plt.plot(fpr, tpr, lw=2.5, label=f'AUC = {auc(fpr, tpr):.3f}')
plt.plot([0, 1], [0, 1], '--', color='gray')
plt.xlabel('誤検出率 FPR'); plt.ylabel('検出率 TPR'); plt.legend()
plt.tight_layout(); plt.show()

## 目標に答えられたか

- 今日の目標は「心臓病かどうかを、見逃しを抑えて判定したい」でした
- TODO①の混同行列で、見逃し（左下）は何件でしたか？
- TODO②で、しきい値を0.5から0.3に下げると、再現率（見逃しの少なさ）はどう変わりましたか？ 適合率は？
- 医療のこの問題では、適合率と再現率のどちらを重視すべきでしょうか？ その理由は？
- ROC曲線は左上に近かったですか？ AUCはいくつでしたか？

## 課題（提出）

**提出するもの**: 応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

4節では、しきい値を 0.5 と 0.3 の二つだけ試しました。今度は 0.1, 0.2, …, 0.9 の9通りを for 文で回し、それぞれの適合率と再現率を計算して、表（DataFrame）で表示してください。
各しきい値の結果を辞書にしてリストに貯め、最後に `pd.DataFrame` に渡すと表になります。
`precision_score` と `recall_score` には `zero_division=0` を付けます（陽性と判定した人がゼロのとき警告を出さないためです）。使う変数は4節の `prob` と `yte` です。


In [ ]:
rows = []
for t in [i / 10 for i in range(1, 10)]:     # 0.1, 0.2, ..., 0.9
    p = (prob > t).astype(int)
    rows.append({'しきい値': t,
                 '適合率': precision_score(yte, p, zero_division=0),
                 '再現率': recall_score(yte, p, zero_division=0)})
tab = pd.DataFrame(rows)
tab.round(2)


<details><summary>詰まったら</summary>

しきい値のリストは `[i / 10 for i in range(1, 10)]` で作れます（`np.arange(0.1, 1.0, 0.1)` は 0.30000000000000004 のような値になるので避けます）。
各しきい値で辞書 `{'しきい値': t, '適合率': ..., '再現率': ...}` をリストに追加し、最後に `pd.DataFrame(リスト)` に渡します。

</details>


### 応用②（判断）

応用①の表で、適合率 0.8 以上を保てるしきい値のうち、**最小のもの**はいくつですか。
0.1 刻みの値を小数第1位まで（例: 0.6）で答えてください。


In [ ]:
ok = tab[tab['適合率'] >= 0.8]
print('答え:', ok['しきい値'].min())


<details><summary>詰まったら</summary>

表を `tab` とすると、`tab[tab['適合率'] >= 0.8]` で条件を満たす行だけが残ります。その `'しきい値'` 列の `.min()` が答えです。

</details>


### 応用③（解釈）

見逃し（再現率）と誤検出（適合率）のどちらを重く見るべきか、その理由と、応用①の表からどのしきい値を勧めるかを、診察にあたる医師に向けて3行で書いてください。


**模範例**

見逃しを重く見るべきです。病気の人を「健康」と帰すと治療の機会を失いますが、健康な人を「疑いあり」とするのは再検査で済むからです。

表では、しきい値 0.3 で再現率 1.00（見逃しゼロ）、適合率 0.70。0.4 に上げると適合率は 0.81 に戻りますが、再現率が 0.93 に下がり、28人中2人を見逃します。

私はしきい値 0.3 を勧めます。誤検出は健康な33人中12人に増えますが、このテストでの見逃しゼロと引き換えなら再検査の負担のほうが受け入れやすいと考えます。


<details><summary>詰まったら</summary>

しきい値を下げると見逃しが減り、誤検出が増えます。「間違えたときに何が起きるか」を両方の側で一言ずつ書くと、勧める理由になります。

</details>


## 発展（任意）

### キャリブレーション曲線——その確率は信じてよいか

ここまで「確率 0.3 で区切る」「0.4 なら適合率 0.81」と議論してきました。でもこの議論は、`predict_proba` の 0.3 が本当に「30%」であってこそ成り立ちます。モデルの出す確率が、実際に病気だった割合と合っているかを**キャリブレーション（較正）**と呼びます。

確かめ方は単純です。予測確率で患者を区間に分け、区間ごとに「実際に病気だった割合」を数えて、予測確率の平均と並べます。点が対角線に乗れば、その確率は信じてよい、ということです。

ロジスティック回帰は確率そのものを当てはめる手法なので、比較的よく合います。一方、ランダムフォレストのように多数の木の投票の割合を確率にする手法や、サポートベクターマシンは、確率が歪みやすいことが知られています。歪んでいる場合は `CalibratedClassifierCV` で補正できます。


In [ ]:
from sklearn.calibration import CalibrationDisplay

disp = CalibrationDisplay.from_estimator(clf, Xte, yte, n_bins=5, name='LogisticRegression')
plt.title('キャリブレーション曲線（テスト61人）')
plt.tight_layout(); plt.show()


In [ ]:
# 区間ごとの人数も見る（61人なので、中央の区間は人数が少ない）
grp = pd.cut(prob, bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0])
pd.DataFrame({'予測確率': prob, '実際': yte}).groupby(grp, observed=True).agg(
    人数=('実際', 'size'), 予測確率の平均=('予測確率', 'mean'), 実際に病気の割合=('実際', 'mean')).round(2)


**読み方**　横軸が予測確率の平均、縦軸がその区間で実際に病気だった割合です。点線の対角線に近いほど、確率が「言葉どおり」だと読めます。

両端の区間はよく合っています。予測の平均 0.08 の区間（18人）で実際は 0.00、予測の平均 0.94 の区間（23人）で実際は 0.96 です。

中央の3区間は人数が 11人・2人・7人しかなく、点が大きく揺れます。たとえば予測 0.6〜0.8 の区間は 7人中3人が病気で 0.43 と、対角線を下回っています。ただし7人の3人が4人に変わるだけで 0.57 になる程度の揺れなので、この区間で「確率が歪んでいる」と言い切るのは早計です。

曲線は人数の多い区間を信じ、少ない区間は割り引いて読みます。本気で確かめるなら、テスト61人ではなく `cross_val_predict` で303人全員の確率を作ってから描くと安定します。`from sklearn.model_selection import cross_val_predict` のうえで、`cross_val_predict(pipe(LogisticRegression(max_iter=5000)), X, y, cv=5, method='predict_proba')[:, 1]` とすると、各人を「その人を含まない4/5で学習したモデル」で予測した確率が得られます。

試すなら、`from sklearn.ensemble import RandomForestClassifier` のうえで `pipe(RandomForestClassifier(random_state=42)).fit(Xtr, ytr)` を作り、`CalibrationDisplay.from_estimator(..., ax=disp.ax_)` で同じ図に重ねてみてください。
